# Introduction

In this notebook, we first introduce the high-level architecture of NVIDIA FLARE. Then, we walkthrough core user APIs of NVIDIA FLARE which allow scientists and developers to quickly build fundamental components of federated learning and adapt any centralized computation routine to federated paradigm. Finally, we finish this notebook by putting all elements together with an example of federated numerical computation using `numpy`. The example uses FL Simulator, a handy runtime tool that allows researchers to quickly test-run a federated job on a local development PC.


# NVIDIA FLARE Architecture

The diagram below summarizes the high-level architecture of NVIDIA FLARE.

<img src="images/nvflare-arch.png" alt="NVFLARE Arch" width=400/>

Now let's look at this diagram in more details. In NVIDIA FLARE, a federated workflow is centered around the interaction between server-side "Controller" and client-side "Executors", through the concept of "tasks", which can be local training, local validation, or any other general routines that happen on the client side. The interaction between server and client is defined and conceptualized as a "Federated Job". NVIDIA FLARE provides "Runtime" to run different jobs. More details on these fundamental components:
- Server-side Controller: a Controller defines the server-side workflow and coordinates with client-side Executors through task assignments. NVIDIA FLARE provides server-side Controller classes with extensible APIs which allow developers to re-use classic federated workflows (FedAvg, FedOpt, etc.) or efficiently implement customized workflows.
- Client-side Executor: the client-side Executors recieve tasks from the server-side Controller, and execute them locally for each client. In NVIDIA FLARE, client-side Executors can be created using carefully designed intuitive APIs, which allow researchers to easily convert an existing centralized computation / training codes to a federated / distributed paradigm.
- Job: NVIDIA FLARE provides `FedJob` class, an abstraction of server-client interaction, which allow users to set up and configure a federated workflow. A `FedJob` can be exported and run by NVIDIA FLARE runtime.
- Runtime: NVIDIA FLARE provides runtime backends to run federated jobs, with capabilities of monitoring and managing multiple jobs. An example of this is the FL Simulator, which allows researchers to test-run federated jobs on a local development PC before real-world deployment.

Notice also the possibility of adding filters to task data and / or results, at any moment of the Controller & Executor interaction. This filtering mechanism provides a flexible way to add security & privacy filters, for example homomorphic encryption and differential privacy filters.


# Core User APIs

NVIDIA FLARE offers a rich set of APIs. In this notebook, we look at the core user APIs that allow you to adapt a centralized compute routine to federated paradigm in 5 minutes. These APIs can be summarized in 3 categories:
- Server-side APIs: these are essentially APIs for implementing server-side Controllers. In particular, we will look at the [ModelController](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L23), and show you how you can easily re-use classic federated workflow, or customize your own workflow.
- Client-side APIs: these are essentially APIs for implementing client-side Executors. We will look at NVIDIA FLARE [Client APIs](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api.html#client-api), a suite of carefully designed APIs that allow user to easily adapt any centralized compute to federated compute.
- Job APIs: these are essentially APIs for creating a Federated Job, to be run using NVIDIA FLARE runtime. More specifically, we will look at the [FedJob](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L147) class.

In the following section, we will demonstrate the use of these core APIs with a simple `numpy` example.


# Example: Federated `numpy`

This example is taken from NVIDIA FLARE's [example repository](https://github.com/NVIDIA/NVFlare/tree/109da964126c015d248718dbff7865206aa2ad6f/examples/hello-world/hello-fedavg-numpy). In this simple example, we will implement the following workflow:
- Client-side: for each round, the clients will perform simple arithmetics on a fixed `numpy` array 
- Server-side: the server will simply get the average value of the arrays sent from clients

We will show how you can easily implement this example using NVIDIA FLARE's server, client and job APIs.


### Setup

Let's first copy the example to our notebook workspace

In [4]:
!if [ ! -d examples ]; then mkdir examples; fi
!if [ ! -d examples/hello-fedavg-numpy ]; then \
    cp -r ../NVFlare/examples/hello-world/hello-fedavg-numpy examples/hello-fedavg-numpy; fi
!tree examples/hello-fedavg-numpy

83745.96s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
83751.08s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
83756.20s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


examples/hello-fedavg-numpy
├── README.md
├── fedavg_script_runner_hello-numpy.py
├── hello-fedavg-numpy_flare_api.ipynb
├── hello-fedavg-numpy_getting_started.ipynb
├── requirements.txt
└── src
    └── hello-numpy_fl.py

2 directories, 6 files


### Client-Side Implementation

Let's first look at how to implement client-side logic. Client-side implementation is in file: [examples/hello-fedavg-numpy/src/hello-numpy_fl.py](examples/hello-fedavg-numpy/src/hello-numpy_fl.py). 

Let's first remove all NVIDIA FLARE related code. After that, we have the following: 
```python
import copy
import numpy as np

def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1

def evaluate(input_arr):
    # mock evaluation metrics
    return np.mean(input_arr)

def main():
    # Get a "mock" model: input_model
    input_model = ...
    
    if input_model.params == {}:
        params = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
    else:
        params = np.array(input_model.params["numpy_key"], dtype=np.float32)
        
    # training
    new_params = train(params)
    # evaluation
    metrics = evaluate(params)

if __name__ == "__main__":
    main()
```

You can see that this is a very straight-foward computation:
- We first get a `input_model` object, whose `params["numpy_key"]` attribute is a `numpy` array. The way how we get the `input_model` object is omitted and is not important in this example. If `input_model.params["numpy_key"]` is empty, the default value given to it is the array `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]`. But `input_model.params["numpy_key"]` might already has value, if it is for example loaded from a file.
- The `train()` function performs a very simple task: adding value `1` to each element of `input_model.params["numpy_key"]`
- The `evaluate()` function perform a "mock" evaluation, by simply computing the mean value of `input_model.params["numpy_key"]`

Now let's make this simple computation federated: we will show you how easy it is to do that using NVIDIA FLARE's client APIs.

First we need to import NVIDIA FLARE, by simply doing
```python
import nvflare.client as flare
```
And before doing anything, it is important to perform necessary initialization by calling:
```python
flare.init()
```
Now let's look at the `main()` function above: this is where the computation logic happens. 

The first code block in the original `main()` function above is getting the `input_model`:
```python
# Get a "mock" model: input_model
input_model = ...
```
However, in a federated paradigm, a client does not need to worry about how to get a model, it simply receives a copy of the global model from the server. With NVIDIA FLARE, this is done using the `receive()` API. Therefore we can replace this first code block by:
```python
input_model = flare.receive()
```
This tells the client that it will, at some point, receive a model from the server. As of when and how, that is the server's concern. As of the format of the `input_model` received from the server, NVIDIA FLARE uses a standard class [`flare.FLModel`](https://nvflare.readthedocs.io/en/main/programming_guide/fl_model.html#flmodel), which is a dictionary-like seriablizable class.

The rest of the code blocks in the `main()` function stays the same: we set default value to `input_model.params["numpy_key"]` if it's empty. Then we perform our simple `train()` operation (adding `1` to `input_model.params["numpy_key"]`) and `evaluate()` operation (compute mean of `input_model.params["numpy_key"]`).

This is something missing in the end of the `main()` function though. Remember that in a federated paradigm, a client interacts with the server constantly, and needs to send the local computation results to the server periodically. To do that, NVIDIA FLARE provides a convenient API `flare.send()`. As of the format of the results to be sent to the server, similar to `flare.receive()`, NVIDIA FLARE the standard `flare.FLModel` class. Adding the following in the end of the `main()` function, we send the computation results, a.k.a the modified `new_params` in our case, to the server:
```python
output_model = flare.FLModel(
            params={"numpy_key": new_params},
            params_type="FULL",
            metrics={"accuracy": metrics},
            current_round=input_model.current_round,
        )
flare.send(output_model)
```
Notice that appart from the `params` argument which refers to the parameters of the model / computation results, `flare.FLModel` has many other input arguments, such as `param_type`, `metrics`, `current_round`. Please refer to its [API documentation](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.app_common.abstract.fl_model.html#module-nvflare.app_common.abstract.fl_model) for more detailed explanation of these arguments.

At this point, we are almost finished on the client side, but there is still one last element missing. Remember that a federated workflow is usually an iterative interaction process between server and clients. The server decides on how many iterations, or "rounds" of computations to be performed by each client: this needs to be handled on the client side. With NVIDIA FLARE, this is done by wrapping the client computation inside a `while` clause:
```python
while flare.is_running():
    # client computation code
    ...
```
As the name indicates, the computation wrapped inside the `while flare.is_running():` will be performed for as many iterations as required by the server.

Now putting all things together, we have the final federated client-side implementation:

```python
import copy
import numpy as np
import nvflare.client as flare # Import flare

def train(input_arr):
    output_arr = copy.deepcopy(input_arr)
    # mock training with plus 1
    return output_arr + 1

def evaluate(input_arr):
    # mock evaluation metrics
    return np.mean(input_arr)

def main():
   
    flare.init() # Initialization
    
    while flare.is_running(): # Run iteratively
        
        input_model = flare.receive() # Get copy of model from server

        if input_model.params == {}:
            params = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
        else:
            params = np.array(input_model.params["numpy_key"], dtype=np.float32)

        # training
        new_params = train(params)
        # evaluation
        metrics = evaluate(params)

        # Send results to server
        output_model = flare.FLModel(
            params={"numpy_key": new_params},
            params_type="FULL",
            metrics={"accuracy": metrics},
            current_round=input_model.current_round,
        )

        flare.send(output_model)

if __name__ == "__main__":
    main()
```

If you compare the federated implementation with the original one shown in the beginning of this section, you can see that with just the addition of a few APIs, we have adapted a centralized compute routine to a federated one. Though the computation is this example is quite trivial, this type of adaptation can be done on any more complex operations, as we will in later content of this course.

### Server-Side Implementation

Now, let's look at how to implement the server-side workflow. As we've seen from the NVIDIA FLARE architecture, this involves creating Controllers. The server-side implementation is in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py). Below is the part of code that is related to server-side implementation:
```python
n_clients = 2
num_rounds = 3

persistor_id = job.to_server(NPModelPersistor(), "persistor")

# Define the controller workflow and send to server
controller = FedAvg(
    num_clients=n_clients,
    num_rounds=num_rounds,
    persistor_id=persistor_id,
)
```
In this example, the server-side workflow is quite straight-forward: computing the average from results received from clients. This corresponds exactly to the [FedAvg (Federated Averaging) Controller](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L18) that is already implemented in NVIDIA FLARE. The FedAvg Controller is instantiated with 3 arguments:
- `num_clients`: the number of clients to select for each training round. Notice that, this is different than the actual total number of participating clients. This parameter indicates the number of clients that will participate in each aggregation round. `num_clients` cannot be greater than the number of total clients, but it can be smaller, in which case, a random subset of `num_clients` clients will be selected for each round.
- `num_rounds`: number of rounds
- `persistor_id`: ID to a `persistor` object

A `persistor` in NVIDIA FLARE is in charge of loading & serializing something, in this context, the global model on the server side. In this example, the global model is just a `numpy` array, therefore we use the class [`NPModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/np/np_model_persistor.py#L39) provided by NVIDIA FLARE. The `NPModelPersistor` implements:
- Default array initialization: this is typically for the initial round of a federated workflow, where the server needs to send a default global array to the client.
- `save_model()`: for serializing the array to the filesystem as a numpy `.npy` file.
- `load_model()`: for loading the array from the filesystem.

The ID of the created `persistor` is passed to the FedAvg Controller, so that when the Controller needs to initialze the global model, saving the model to disk or loading a model from disk, it will call the corresponding function from the `persistor`.

NVIDIA FLARE provides many other default model persistors, for instance the [`PTFileModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/pt/file_model_persistor.py#L36) for `Pytorch` models, [`TFModelPersistor`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/tf/model_persistor.py#L26) for Tensorflow models, etc. You also have the possibility to write your own model persistor by sub-classing the [ModelPersistor](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/abstract/model_persistor.py#L26) class.

Now, let's look into the details of the FedAvg Controller. The FedAvg class inherits from the [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L29) class, which itself inherits from the [ModelController](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/model_controller.py#L23) class:

The `ModelController` class is the fundamental Controller class, that allows developers to implement any custom server-side workflows. This class has the following key APIs:
  - `run()` method: this is an asbtract function that must be overridden by all Controller classes that inherit from ModelController. This is the key function where the server-side workflow logic is actually implemented.
  - `load_model()` and `save_model()`: these are methods implementing how to load and save model on the server-side, including initializing the first model. As a matter of fact, these methods actually call their corresponding function in the `persistor` object that is passed to the `ModelController`. We will look at the `persistor` object later.
  - `sample_clients()`: this function returns a list of active clients.
  - `send_model()` and `send_model_and_wait()`: these functions implement communications between server and client via tasks.

We can see how the design of `ModelController` offers a flexible and easy way to custom any server-side workflow. In practice, developers typically do not need to worry about low-level communications that are handled by `send_model()` and `send_model_and_wait()`. All they need to re-implement is workflow-related logic:
  - the `run()` function
  - model loading & saving
  - and any other custom routines needed by the workflow

Let's take Federated Averaging (FedAvg class & BaseFedAvg class) as an example. The `run()` method is implemented in the [FedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/fedavg.py#L34) class:
 ```python
    def run(self) -> None:
        model = self.load_model()
        model.start_round = self.start_round
        model.total_rounds = self.num_rounds

        for self.current_round in range(self.start_round, self.start_round + self.num_rounds):
            model.current_round = self.current_round
            clients = self.sample_clients(self.num_clients)
            results = self.send_model_and_wait(targets=clients, data=model)
            aggregate_results = self.aggregate(results, aggregate_fn=self.aggregate_fn)
            model = self.update_model(model, aggregate_results)
            self.save_model(model)
```
Inside the `run()` function, a classic Federated Averaging workflow is implemented:
- An initial model is loaded with `load_model()` method, handled by the persistor (more on this later).
- For each round, we get a list of active clients via `sample_clients()`, and send a copy of the current model for local training via `send_model_and_wait()`. These are the built-in functions of `ModelController`.
- When local clients return results for each round, we perform model aggregation via `aggregate()` function. The aggregation function is a custom function implemented in class [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L115), and performs a simple weighted average of the clients local training results.
- When the aggregated result is ready, we update the global model via `update_model()` function. The model update function is a custom function implemented in class [BaseFedAvg](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_common/workflows/base_fedavg.py#L148).
- Finally, we save the model for each round, via the `save_model()` function.

As we can see, `ModelController` separate the low-level communications and workflow logics, and allows developers to focus on workflow related implementation. In practice, the `BaseFedAvg` class can be leveraged to implement most of the centralized federated workflows, with the flexibility to easily customize server-side aggregation function and model update function.


### Putting Everything into a Federated Job

Now we have the client- & server-side implementation, all that remains is to put everything together into a Federated Job, so that it can be run by NVIDIA FLARE's runtime. This is done in file [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py) (order of lines modified to facilitate explanation):
```python
from nvflare import FedJob
from nvflare.app_common.np.np_model_persistor import NPModelPersistor
from nvflare.app_common.widgets.intime_model_selector import IntimeModelSelector
from nvflare.app_common.workflows.fedavg import FedAvg
from nvflare.job_config.script_runner import FrameworkType, ScriptRunner

if __name__ == "__main__":

    job = FedJob(name="hello-fedavg-numpy")

    n_clients = 2
    num_rounds = 3
    
    persistor_id = job.to_server(NPModelPersistor(), "persistor")

    # Define the controller workflow and send to server
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
    job.to(controller, "server")

    job.to(IntimeModelSelector(key_metric="accuracy"), "server")

    # Add clients
    train_script = "src/hello-numpy_fl.py"
    for i in range(n_clients):
        executor = ScriptRunner(script=train_script, script_args="", framework=FrameworkType.NUMPY)
        job.to(executor, f"site-{i+1}")

    job.export_job("/tmp/nvflare/jobs/job_config")
    job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```

Let's analyze this file. First, we define a `FedJob` object:
```python
    job = FedJob(name="hello-fedavg-numpy")
```
The [`FebJob`](https://nvflare.readthedocs.io/en/main/programming_guide/fed_job_api.html) class aims to model the federated workflow and server-client interactions, and it offers APIs to Pythonically define and create job configurations.

The next couple of lines are related to server-side Controller, as we've already seen:
```python
    n_clients = 2
    num_rounds = 3
    
    persistor_id = job.to_server(NPModelPersistor(), "persistor")

    # Define the controller workflow and send to server
    controller = FedAvg(
        num_clients=n_clients,
        num_rounds=num_rounds,
        persistor_id=persistor_id,
    )
    job.to(controller, "server")
```
Here we see the usage of `job.to` API: `job.to` sends an object to a specific target, where the target can be a client name or `server`. `job.to_server(...)` is equivalent to `job.to(..., "server")`. Therefore we can interpret the above usage as follows:
- `job.to_server(NPModelPersistor(), "persistor")`: sends an instance of `NPModelPersistor` to the server, give the instance an ID of "persistor"
- `job.to(controller, "server")`: send the FedAvg Controller to the server. This is equivalent to `job.to_server(controller)`

The next line:
```python
job.to(IntimeModelSelector(key_metric="accuracy"), "server")
```
sends another component, `IntimeModelSelector` to the server. We would not go into details of this component in this course, but all you need to know if, this component allows the server to select & save the best global model during federated training rounds, based on specific metric, which in this case is the "accuracy".  This is possible since each client send an `FLModel` object to the server, which has an attribute `metrics` that the `IntimeModelSelector` can refer to. For more details, please check out the [documentation](https://nvflare.readthedocs.io/en/main/programming_guide/component_configuration.html#component-configuration-and-event-handling). In this example, the model selection based on a "mock" evaluation accuracy does not really make sense, but we will see its better use-case later in this course with other more realistic examples.

Next, we have a couple of lines that refer to the client-side implementation:
```python
    # Add clients
    train_script = "src/hello-numpy_fl.py"
    for i in range(n_clients):
        executor = ScriptRunner(script=train_script, script_args="", framework=FrameworkType.NUMPY)
        job.to(executor, f"site-{i+1}")
```
Here, we are essentially turning our client-side Python script [`examples/hello-fedavg-numpy/src/hello-numpy_fl.py`](examples/hello-fedavg-numpy/src/hello-numpy_fl.py) into an Executor object using the `ScriptRunner` API, and send it to each of the clients using `job.to` API. Most of the magic here is done by [`ScriptRunner`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/script_runner.py#L30), which alleviates the need to manually writing custom Executors. Instead, developers can start from a centralized training script, adapt it to a federated client-side local training script, and then convert it to an Executor using `ScriptRunner`.

Now the definition & configuration of a Federated Job is complete, we can export it to a local folder and run it later using different runtime backends:
```python
job.export_job("/tmp/nvflare/jobs/job_config")
```

Alternatively, we can run it directly with FL Simulator using `FedJob` API:
```python
job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```
The [FL Simulator](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_cli/fl_simulator.html) is a handy runtime designed for researchers to quickly test-run the federated job on a single development PC. The first argument to [`job.simulator_run()`](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496) is a directory to run the federated job and save results. the function also accepts multiple other arguments, including total number of clients, number of threads, GPU index etc. See [here](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/job_config/api.py#L496) for details.

Noted that FL Simulator is designed for convenience, therefore it does not include necessary security features for real-world deployment. We will cover real-world deployment in later chapters of this course.


### Running the Job with FL Simulator

You can run the job in FL Simulator by simply running the python script [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py):


In [4]:
!python3 examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py

2024-09-06 23:20:08,476 - SimulatorRunner - INFO - Create the Simulator Server.
2024-09-06 23:20:08,477 - CoreCell - INFO - server: creating listener on tcp://0:42787
2024-09-06 23:20:08,491 - CoreCell - INFO - server: created backbone external listener for tcp://0:42787
2024-09-06 23:20:08,491 - ConnectorManager - INFO - 5185: Try start_listener Listener resources: {'secure': False, 'host': 'localhost'}
2024-09-06 23:20:08,491 - nvflare.fuel.f3.sfm.conn_manager - INFO - Connector [CH00002 PASSIVE tcp://0:60629] is starting
2024-09-06 23:20:08,992 - CoreCell - INFO - server: created backbone internal listener for tcp://localhost:60629
2024-09-06 23:20:08,992 - nvflare.fuel.f3.sfm.conn_manager - INFO - Connector [CH00001 PASSIVE tcp://0:42787] is starting
2024-09-06 23:20:09,034 - nvflare.fuel.hci.server.hci - INFO - Starting Admin Server localhost on Port 36029
2024-09-06 23:20:09,035 - SimulatorRunner - INFO - Deploy the Apps.
2024-09-06 23:20:09,039 - SimulatorRunner - INFO - Create 

There is quite a lot of console output, but you can essentially pick up a couple of prints, indicating that the initial `numpy` array `[[1,2,3], [4,5,6], [7,8,9]]` is incremented by 1 to each of its values after each round, and becoming `[[4,5,6], [7,8,9], [10,11,12]]` after a total of 3 rounds.


Alternatively you can also run the job using NVIDIA FLARE's CLI tool for FL Simulator. To do that, you should first export the job to a local folder. You probably need to uncomment the line:
```python
job.export_job("/tmp/nvflare/jobs/job_config")
```
and comment the line:
```python
job.simulator_run("/tmp/nvflare/jobs/workdir", gpu="0")
```
in [examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py](examples/hello-fedavg-numpy/fedavg_script_runner_hello-numpy.py)

Then execute the following command:

In [ ]:
!nvflare simulator -h

In [ ]:
!mkdir /tmp/hello-numpy-workspace
!nvflare simulator -w /tmp/hello-numpy-workspace -n 2 -t 2 -gpu 0 /tmp/nvflare/jobs/job_config

Watch the output above for the server to signal the run has completed:
```
    SimulatorServer - INFO - shutting down server
    SimulatorServer - INFO - canceling sync locks
    SimulatorServer - INFO - server off
```


We can then check the contents of the `hello-numpy-cross-val-workspace` directory to see the job output.